In [1]:
import ir_datasets
from nltk.stem import WordNetLemmatizer
import nltk
from helper import create_tokenized_word_list

dataset = ir_datasets.load("wikir/en1k/training")
tokenized = create_tokenized_word_list(dataset)

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/tahas44/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
nltk.download("punkt_tab")
nltk.download("wordnet")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/tahas44/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /home/tahas44/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [3]:
lemmatized_tokenized_docs = []
lematizer = WordNetLemmatizer()

for query_list in tokenized:
    lemmatized_words = [lematizer.lemmatize(word) for word in query_list]
    lemmatized_tokenized_docs.append(lemmatized_words)

print("Docs lemmatized")

Docs lemmatized


In [4]:
from helper import create_tokenized_word_list_for_query
query_tokenized = create_tokenized_word_list_for_query(dataset)
print("Queries tokenized!")

Queries tokenized!


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/tahas44/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
lemmatized_tokenized_queries = []

for query_list in query_tokenized:
    lemmatized_words = [lematizer.lemmatize(words) for words in query_list]
    lemmatized_tokenized_queries.append(lemmatized_words)

print("Queries lemmatized")

Queries lemmatized


In [6]:
from rank_bm25 import BM25Okapi
bm25 = BM25Okapi(lemmatized_tokenized_docs)

In [7]:
scores_of_all_queries = []

for query in lemmatized_tokenized_queries:
    scores_of_all_queries.append(bm25.get_scores(query))

In [8]:
from collections import defaultdict
from helper import Scoredoc

doc_dict = defaultdict(str)

for i, doc in enumerate(dataset.docs_iter()):
    doc_dict[i] = doc.doc_id

doc_dict = dict(doc_dict)

qrels_dict = defaultdict(list)

for qrel in dataset.qrels_iter():
    qrels_dict[qrel.query_id].append(qrel.doc_id)

qrels_dict = dict(qrels_dict)

score_doc_dict = defaultdict(list)

for scoreddoc in dataset.scoreddocs_iter():
    doc_id = scoreddoc.doc_id
    score = scoreddoc.score

    scoreddoc_object = Scoredoc(doc_id, score)

    score_doc_dict[scoreddoc.query_id].append(scoreddoc_object)

score_doc_dict = dict(score_doc_dict)

print("Necessary dicts created!")

Necessary dicts created!


In [9]:
import pandas as pd
query_ids = [query.query_id for query in dataset.queries_iter()]
df = pd.DataFrame(query_ids, columns=["Query_ID"])
df

,Query_ID
0,123839
1,188629
2,13898
3,316959
4,515031
...,...
1439,896124
1440,12319
1441,4421
1442,296526


In [10]:
from helper import create_AP, create_ndcg, create_statistical_columns
df = create_statistical_columns(df, qrels_dict, doc_dict, scores_of_all_queries)
df = create_AP(df, qrels_dict, doc_dict, scores_of_all_queries)
df = create_ndcg(df, doc_dict, scores_of_all_queries, score_doc_dict)

In [11]:
df

,Query_ID,recall_5,recall_10,precision_5,precision_10,f_score_5,f_score_10,AP_5,AP_10,NDCG_5,NDCG_10
0,123839,66.666667,100.000000,80.0,60.0,72.727273,75.000000,0.591667,0.873413,1.000000,1.000000
1,188629,33.333333,33.333333,40.0,20.0,36.363636,25.000000,0.333333,0.333333,0.889669,0.918338
2,13898,33.333333,33.333333,40.0,20.0,36.363636,25.000000,0.333333,0.333333,0.000000,0.000000
3,316959,11.111111,22.222222,20.0,20.0,14.285714,21.052632,0.111111,0.148148,1.000000,0.998975
4,515031,7.142857,14.285714,20.0,20.0,10.526316,16.666667,0.014286,0.034694,0.546387,0.518722
...,...,...,...,...,...,...,...,...,...,...,...
1439,896124,12.500000,12.500000,20.0,10.0,15.384615,11.111111,0.125000,0.125000,0.464253,0.502333
1440,12319,4.545455,4.545455,20.0,10.0,7.407407,6.250000,0.045455,0.045455,0.197103,0.209197
1441,4421,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.495470,0.468331
1442,296526,0.000000,10.000000,0.0,10.0,0.000000,10.000000,0.000000,0.010000,0.328526,0.325590


In [12]:
mydict = {
    "Method": "BM25 - Lemmatization",
    "recall_5_mean": df["recall_5"].mean(),
    "recall_5_std": df["recall_5"].std(),
    "recall_5_max": df["recall_5"].max(),
    "recall_5_min": df["recall_5"].min(),
    "recall_10_mean": df["recall_10"].mean(),
    "recall_10_std": df["recall_10"].std(),
    "recall_10_max": df["recall_10"].max(),
    "recall_10_min": df["recall_10"].min(),
    "precision_5_mean": df["precision_5"].mean(),
    "precision_5_std": df["precision_5"].std(),
    "precision_5_max": df["precision_5"].max(),
    "precision_5_min": df["precision_5"].min(),
    "precision_10_mean": df["precision_10"].mean(),
    "precision_10_std": df["precision_10"].std(),
    "precision_10_max": df["precision_10"].max(),
    "precision_10_min": df["precision_10"].min(),
    "f_score_5_mean": df["f_score_5"].mean(),
    "f_score_5_std": df["f_score_5"].std(),
    "f_score_5_max": df["f_score_5"].max(),
    "f_score_5_min": df["f_score_5"].min(),
    "f_score_10_mean": df["f_score_10"].mean(),
    "f_score_10_std": df["f_score_10"].std(),
    "f_score_10_max": df["f_score_10"].max(),
    "f_score_10_min": df["f_score_10"].min(),
    "MAP_5": df["AP_5"].mean(),
    "MAP_10": df["AP_10"].mean(),
    "NDCG_5_mean": df["NDCG_5"].mean(),
    "NDCG_5_std": df["NDCG_5"].std(),
    "NDCG_5_max": df["NDCG_5"].max(),
    "NDCG_5_min": df["NDCG_5"].min(),
    "NDCG_10_mean": df["NDCG_10"].mean(),
    "NDCG_10_std": df["NDCG_10"].std(),
    "NDCG_10_max": df["NDCG_10"].max(),
    "NDCG_10_min": df["NDCG_10"].min()
}

In [13]:
df_parquet = pd.DataFrame(mydict, index=[0])
df_parquet

,Method,recall_5_mean,recall_5_std,recall_5_max,recall_5_min,recall_10_mean,recall_10_std,recall_10_max,recall_10_min,precision_5_mean,...,MAP_5,MAP_10,NDCG_5_mean,NDCG_5_std,NDCG_5_max,NDCG_5_min,NDCG_10_mean,NDCG_10_std,NDCG_10_max,NDCG_10_min
0,BM25 - Lemmatization,14.593247,14.569564,83.333333,0.0,20.86522,19.584312,100.0,0.0,29.847645,...,0.116666,0.143757,0.505786,0.35388,1.0,0.0,0.506285,0.350688,1.0,0.0


In [14]:
df_parquet.to_parquet("BM25Lemmatization.parquet")
print("parquet")

parquet
